## Laboratorio 4

- Diego Valenzuela 22309 
- Daniel Dubon 22233
- Nelson García Bravatti 22434
- Joaquin Puente 22296

### Task 1 Diseño

1. Espacio de Estados y Espacio de Acciones

    Espacio de Estados: El almacen es una cuadricula de 8 x 8.

    Eleccioó: gymnasium spaces Discrete 64

    Justificación: Un espacio Discreto 64 mapea cada celda bidimensional a un unico numero entero entre 0 y 63 y esto es ideal y necesario porque tanto SARSA como Q Learning, en sus formas clasicas tabulares, requieren un estado discreto para usarse como indice en la matriz o Q Table de dimensiones 64 x 4.

    Por que no usar Box: La clase Box se utiliza para representar espacios continuos como arreglos de numeros decimales. Si se usara Box el espacio de estados seria infinito y no se podria construir una Q Table clasica y nos veriamos obligados a usar aproximadores de funciones lo cual esta fuera del alcance de una comparacion estandar de SARSA contra Q Learning.

    Espacio de Acciones: Movimientos direccionales del robot.

    Elección: gymnasium spaces Discrete 4

    Justificación: Representa 4 acciones deterministas y mutuamente excluyentes: Arriba 0, Abajo 1, Izquierda 2, Derecha 3.


2. Función de Recompensa

    - Recompensa por entrega exitosa: +100
    - Penalización por zona de congestion: -20 y se aplica cada vez que se pisa o permanece en una celda de congestion.
    - Penalización por paso:  -1 y se aplica en cualquier otra celda transitable.


    Justificación de las magnitudes:

    El objetivo es que el robot llegue a la meta de la forma mas rapida por la penalizacion por paso y segura por la penalización por congestion y aumentar 100 garantiza que llegar a la meta sea el evento dominante y el objetivo global del episodio, El menos 1 fuerza al robot a buscar la ruta mas corta y el menos 20 es lo suficientemente grande como para disuadir al robot de cruzar la congestion tambien cabe resaltar que cruzar una celda de congestion cuesta lo mismo que dar 20 pasos normales.

    Comportamientos indeseables por mala ponderación:

    Si la penalización por congestion es menor o igual a la penalización por paso el robot desarrollara un comportamiento temerario y si rodear la congestion toma 3 pasos extra pero cruzarla cuesta menos puntos el agente preferira atravesar la congestion. Si la penalizacion por paso es cero o positiva el robot perdera el sentido de urgencia y caminara en circulos infinitamente sin incentivo para llegar a la meta. Si la penalización por congestion es excesivamente alta, por ejemplo menos 1000: Podria provocar que durante la exploracion temprana el agente asocie todo el entorno con un peligro extremo y aprenda una politica suboptima de quedarse quieto o chocar contra la pared constantemente para terminar el episodio rapidamente y evitar castigos.

3. Condición de Terminación del Episodio

    El episodio terminara por cualquiera de estas dos condiciones:

    - Terminated o Estado Terminal Natural: El robot alcanza el estado del Punto de Entrega.
    - Truncated o Limite Maximo de Pasos: El episodio alcanza un limite, por ejemplo un maximo de 100 pasos.

    Es apropiado tener un limite maximo de pasos:

    Si, es necesario porque durante las primeras fases de entrenamiento la politica del robot es casi completamente aleatoria y sin un limite de pasos el robot podria entrar en bucles infinitos en zonas sin penalizaciones, o simplemente vagar sin encontrar la meta jamas y el limite asegura que los episodios finalicen, garantizando que el agente experimente repetidas veces el reinicio del entorno y converja matematicamente.

4. Diseño del Mapa 8 por 8

    Para que la diferencia entre una politica agresiva y una politica conservadora sea empiricamente visible diseñamos el mapa inspirandonos en el problema clasico de Cliff Walking.

    Leyenda de las celdas:
    S: Punto de recogida
    G: Punto de entrega
    P: Pasillo transitable
    O: Obstaculo fijo
    C: Zona de congestion

    Esquema de las primeras 4 filas:

    --

    S P P P P P P G

    O C C C C C C P

    P P P P P P P P

    P O O P O O O P

    --

    Analisis de la configuración espacial:

    En este mapa la primera fila representa una ruta directa muy rapida entre el punto de recogida y la meta, sin embargo justo debajo hay una hilera de Zonas de Congestion y si un robot esta en la fila de arriba y toma una decision aleatoria hacia abajo caera en la congestion, existe una segunda ruta segura por debajo de los obstaculos pero toma muchos mas pasos.

    Comportamiento diferencial esperado:

    Con Q Learning Agresivo y Off Policy esperamos que aprenda la funcion de valor optima asumiendo que actuara siempre de forma ambiciosa, en este caso aprendera que la ruta optima es ir en linea recta por la fila de arriba, sin embargo durante el entrenamiento con mucha exploración, su constante curiosidad hara que resbale frecuentemente hacia abajo sufriendo muchas penalizaciones y su politica es optima en la teoria pero altamente riesgosa en la practica.

    Con SARSA conservador y On Policy: Aqui se actualizan sus valores basandose en la accion real que va a tomar incluyendo los movimientos aleatorios de exploracion donde SARSA aprende que caminar por la fila de arriba es peligroso porque hay probabilidad de terminar en la congestión, por lo tanto SARSA convergera hacia una ruta mas larga pero segura, bajando para alejarse de la zona de congestión, minimizando las penalizaciones durante el entrenamiento a costa de dar mas pasos.